# Week 6 — Feature Engineering

* Example of sample features you can engineer: bed/bath ratio, age of property in years
* Adding more detailed geographic layer using school districts: build a more detailed regional feature by spatially joining each property’s coordinates against the CA School District Areas 2024-25 boundaries (https://data.ca.gov/dataset/california-school-district-areas-2024-
25/resource/7dfaf005-58eb-45db-93b1-7aff091b2172)
* Re-train models with the updated feature set.
* Deliverable: Updated notebook + table comparing old vs new feature sets, including the school district layer.

In [47]:
from pathlib import Path
import geopandas as gpd
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.tree import DecisionTreeRegressor

In [49]:
data_dir = Path("/Users/jingying/Documents/IDX Exchange/py/cleaned_data/run_20261014_180636_999980")
district_file = Path("/Users/jingying/Documents/IDX Exchange/py/DistrictAreas2425/DistrictAreas2425.shp")

X_train_old = sparse.load_npz(data_dir / "X_train.npz")
X_text_old = sparse.load_npz(data_dir / "X_test.npz")

# The target files list the row IDs in the same order as the sparse matrices.
y_train = pd.read_csv(data_dir / "y_train.csv", index_col="row_id")["ClosePrice"]
y_test = pd.read_csv(data_dir / "y_test.csv", index_col="row_id")["ClosePrice"]

# Week 5 also scored the subset within the original training price bounds.
range_ids = pd.read_csv(data_dir / "y_test_avm_range.csv", usecols=["row_id"])["row_id"]
test_groups = {
    "Full test rows": np.ones(len(y_test), dtype=bool),
    "Test rows within training price bounds": y_test.index.isin(range_ids),
}

In [51]:
#districts = gpd.read_file(district_file)[["CDCode", "geometry"]]

#def match_districts(cleaned_csv, row_ids):
#    columns = ["row_id", "Latitude", "Longitude", "Latitude_missing", "Longitude_missing"]
#    sales = pd.read_csv(cleaned_csv, usecols=columns, index_col="row_id").loc[row_ids]
#    observed = sales.loc[sales["Latitude_missing"].eq(0) & sales["Longitude_missing"].eq(0)]
#
#    # Build points from longitude (x) and latitude (y), then match the polygon CRS.
#    points = gpd.GeoDataFrame(
#        index=observed.index,
#        geometry=gpd.points_from_xy(observed["Longitude"], observed["Latitude"]),
#        crs="EPSG:4326",
#    ).to_crs(districts.crs)
#
#    matches = points.sjoin(districts, how="inner", predicate="within")
#    codes_by_sale = matches.groupby(level="row_id")["CDCode"].agg(set)
#
#    # Return one collection of district IDs per matrix row, including unmatched sales.
#    return [codes_by_sale.get(row_id, ()) for row_id in row_ids]

#train_codes = match_districts(data_dir / "train_cleaned.csv", y_train.index)
#test_codes = match_districts(data_dir / "test_cleaned.csv", y_test.index)

In [61]:
# Start from observed values so all validation sales have a row in the export.
sales = pd.read_csv(data_dir / "quality_cleaned_history.csv", 
                    index_col="row_id", 
                    parse_dates=["CloseDate"],
                    usecols=["row_id", "CloseDate", "YearBuilt", "BedroomsTotal", "BathroomsTotalInteger", "Longitude", "Latitude"])
locations = sales[["Longitude", "Latitude"]].dropna()

districts = gpd.read_file(district_file)[["CDCode", "geometry"]]

# Build points from longitude (x) and latitude (y), then match the polygon CRS.
points = gpd.GeoDataFrame(
    index=locations.index,
    geometry=gpd.points_from_xy(locations["Longitude"], locations["Latitude"]),
    crs="EPSG:4326",
).to_crs(districts.crs)

# Store all matched district codes in a single field for each sale.
matches = points.sjoin(districts, how="inner", predicate="within")
codes = matches.groupby(level="row_id")["CDCode"].agg(lambda values: " ".join(sorted(set(values))))

# These features depend only on the individual sale and the fixed district map.
year = sales["CloseDate"].dt.year
month = sales["CloseDate"].dt.month
fixed = pd.DataFrame(index=sales.index)
fixed["PropertyAge"] = year - sales["YearBuilt"]
fixed["BedBathRatio"] = sales["BedroomsTotal"] / sales["BathroomsTotalInteger"].replace(0, np.nan)
fixed["MonthSin"] = np.sin(2 * np.pi * month / 12)
fixed["MonthCos"] = np.cos(2 * np.pi * month / 12)
fixed["TimeIndex"] = (year - 2020) * 12 + month - 1
fixed["SchoolDistrictCodes"] = codes.reindex(sales.index).fillna("")
fixed.to_csv(data_dir / "week6_property_features.csv")

district_codes = fixed["SchoolDistrictCodes"].str.split()
train_codes = district_codes.loc[y_train.index]
test_codes = district_codes.loc[y_test.index]

In [62]:
# Learn the district columns from the training sales only.
district_encoder = MultiLabelBinarizer(sparse_output=True)
train_districts = district_encoder.fit_transform(train_codes)

# Keep one row per test sale and encode only districts seen in training.
known_districts = set(district_encoder.classes_)
test_codes_known = [known_districts.intersection(codes) for codes in test_codes]
test_districts = district_encoder.transform(test_codes_known)

X_train_new = sparse.hstack([X_train_old, train_districts], format="csr")
X_test_new = sparse.hstack([X_test_old, test_districts], format="csr")

In [63]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=15, min_samples_leaf=10, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=20, min_samples_leaf=5,max_features=0.7, random_state=42, n_jobs=2),
}

feature_sets = {
    "Old": (X_train_old, X_test_old),
    "New": (X_train_new, X_test_new),
}

results = []
for name, model in models.items():
    for features, (X_train, X_test) in feature_sets.items():
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

        for group, mask in test_groups.items():
            actual = y_test.to_numpy()[mask]
            predicted = predictions[mask]
            ape = 100 * np.abs(actual - predicted) / actual
            results.append({
                "Test sample": group,
                "Model": name,
                "Features": features,
                "R2": r2_score(actual, predicted),
                "MAE ($)": mean_absolute_error(actual, predicted),
                "MAPE (%)": np.mean(ape),
                "MdAPE (%)": np.median(ape),
            })

In [64]:
comparison = pd.DataFrame(results).pivot(
    index=["Test sample", "Model"],
    columns="Features",
    values=["R2", "MAE ($)", "MAPE (%)", "MdAPE (%)"],
)
comparison = comparison.reindex(["Old", "New"], axis=1, level="Features")
comparison.columns = [f"{metric} {features}" for metric, features in comparison.columns]
comparison.insert(2, "R2 change (New - Old)", comparison["R2 New"] - comparison["R2 Old"])

comparison.to_csv(data_dir / "week6_feature_comparison.csv")
display(comparison.round(4))

R2 Old  R2 New  \
Test sample                            Model                               
Full test rows                         Decision Tree      0.0085  0.0085   
                                       Linear Regression  0.0091  0.0091   
                                       Random Forest      0.0091  0.0092   
Test rows within training price bounds Decision Tree      0.8389  0.8408   
                                       Linear Regression  0.8034  0.8131   
                                       Random Forest      0.8808  0.8825   

                                                          R2 change (New - Old)  \
Test sample                            Model                                      
Full test rows                         Decision Tree                     0.0000   
                                       Linear Regression                 0.0000   
                                       Random Forest                     0.0001   
Test rows within training price bounds Decision Tree                     0.0019   
                                       Linear Regression                 0.0097   
                                       Random Forest                     0.0018   

                                                          MAE ($) Old  \
Test sample                            Model                            
Full test rows                         Decision Tree      429967.1330   
                                       Linear Regression  490192.7551   
                                       Random Forest      398892.3720   
Test rows within training price bounds Decision Tree      200406.3544   
                                       Linear Regression  260169.4344   
                                       Random Forest      170242.4302   

                                                          MAE ($) New  \
Test sample                            Model                            
Full test rows                         Decision Tree      429124.9010   
                                       Linear Regression  482113.6384   
                                       Random Forest      398100.3009   
Test rows within training price bounds Decision Tree      199611.4179   
                                       Linear Regression  252942.2539   
                                       Random Forest      169599.5174   

                                                          MAPE (%) Old  \
Test sample                            Model                             
Full test rows                         Decision Tree           15.9589   
                                       Linear Regression       25.0358   
                                       Random Forest           13.6475   
Test rows within training price bounds Decision Tree           15.0903   
                                       Linear Regression       24.2702   
                                       Random Forest           12.7848   

                                                          MAPE (%) New  \
Test sample                            Model                             
Full test rows                         Decision Tree           15.9517   
                                       Linear Regression       24.0696   
                                       Random Forest           13.6124   
Test rows within training price bounds Decision Tree           15.0871   
                                       Linear Regression       23.3638   
                                       Random Forest           12.7484   

                                                          MdAPE (%) Old  \
Test sample                            Model                              
Full test rows                         Decision Tree            10.4035   
                                       Linear Regression        17.5422   
                                       Random Forest             8.4591   
Test rows within training price bounds Decision Tree 